We consider a bernoulli likelihood. That is, $X^{(1)}, \dots, X^{(n)} \mid \theta \overset{i.i.d.}{\sim} \text{Categorical}(\theta)$, where $\theta = (\theta_0, \theta_1, \dots, \theta_{J-2})$. We encode each categorical $X$ as
$$
X = (X_0, X_1, \dots, X_{J-1}) \in \{ (1, 0, \dots, 0, 0), \dots, (0, 0, \dots, 0, 1) \}
$$
so $X$ is a vector of length $J$ and exactly one element equals to $1$. The likelihood is
$$
p(X \mid \theta) = \left(\prod_{j=0}^{J-2} \theta_j^{X_j} \right) \cdot \left(1-\sum_{j=0}^{J-2} \theta_j \right)^{X_{J-1}}
$$

Denote the $j$-th element of $\theta$ as $\theta_j$ and the $j$-th element of $X$ as $X_j$. Then, the true score is
$$
\nabla_\theta \log p(X \mid \theta) = \left(\frac{X_j}{\theta_j} - \frac{X_{J-1}}{1 - \sum_{j=0}^{J-2}\theta_j} \right), \quad j=0, \dots, J-2
$$
We use a proposal
$$
(\theta_0, \dots, \theta_{J-2}, 1-\sum_{j=0}^{J-2} \theta_j) \sim \text{Dirichlet}(\alpha), \quad \alpha = (\alpha_0, \dots, \alpha_{J-1})
$$
and the proposal score is that for $j=0, \dots, J-2$:
$$
\nabla_\theta \log p(\theta) = \frac{\alpha_j - 1}{\theta_j} - \frac{\alpha_{J-1} - 1}{1 - \sum_{j=0}^{J-2} \theta_j}
$$

**True posterior**:
$$
(\theta_0, \dots, \theta_{J-2}, 1-\sum_{j=0}^{J-2} \theta_j) \mid X^{(1)}, \dots, X^{(n)} \sim \text{Dirichlet}(\alpha_0 + \sum_{i=1}^n X_0^{(i)}, \dots, \alpha_{J-1} + \sum_{i=1}^n X_{J-1}^{(i)})
$$

In [ ]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import time
from tqdm import tqdm
from itertools import cycle
from torch.autograd.functional import jacobian
import copy
from itertools import cycle
import math
import pandas as pd
from scipy.stats import truncnorm
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
def set_seed(seed):
    np.random.seed(seed)  # NumPy RNG
    torch.manual_seed(seed)  # PyTorch CPU RNG
    torch.cuda.manual_seed(seed)  # Current GPU RNG
    torch.cuda.manual_seed_all(seed)  # All GPUs (if using DataParallel or multi-GPU)

    torch.backends.cudnn.deterministic = True  # Use deterministic algorithms
    torch.backends.cudnn.benchmark = False     # Disable autotuner for reproducibility

In [ ]:
set_seed(42)

In [ ]:
J = 3 # J categories
obs_size = 1000 

## Generate training data

In [ ]:
alpha = 3 * torch.ones(J)
print(alpha)

In [ ]:
tr_size = 600
sample_size = 2 * tr_size
theta_r0 = torch.distributions.Dirichlet(alpha).sample((sample_size,))

x_r0 = torch.distributions.Categorical(probs=theta_r0).sample((obs_size, )).T
x_r0 = torch.nn.functional.one_hot(x_r0, num_classes=J).to(theta_r0.dtype).reshape(sample_size, -1)

theta_tr, x_tr = theta_r0[:tr_size, :(J-1)], x_r0[:tr_size]
theta_val, x_val = theta_r0[tr_size:, :(J-1)], x_r0[tr_size:]

## Neural Network Function

In [ ]:
# The neural network, 1-layer ELU() activation
class ELU_LikeScoreMatchingNN(nn.Module):
    def __init__(self, theta_dim, x_dim, obs_size, hidden_size, num_layers = 1):
        super(ELU_LikeScoreMatchingNN, self).__init__()
        
        layers = [nn.Linear(theta_dim + x_dim, hidden_size), nn.ELU()]

        # Add hidden layers based on num_layers
        for _ in range(num_layers - 1):
            layers.append(nn.Linear(hidden_size, hidden_size))
            layers.append(nn.ELU())

        # Output layer to match the desired output size
        layers.append(nn.Linear(hidden_size, theta_dim))

        # Combine all layers into a sequential model
        self.layers = nn.Sequential(*layers)
        self.obs_size = obs_size
        self.x_dim = x_dim

    def forward(self, theta, x):
        if len(theta.shape) == 1: # if one-dimensional
            theta = theta.view(-1, 1)

        if len(x.shape) == 1:
            x = x.view(-1, 1)
        x_dim = int(x.shape[1] / self.obs_size)
        score = self.layers( torch.cat( (theta.repeat_interleave(self.obs_size, dim = 0), x.reshape(-1, x_dim)), dim = 1 ) ).view(theta.shape[0], self.obs_size, theta.shape[1]).sum(dim = 1)
        return score

    def cal_penalty(self, theta, x):
        if len(theta.shape) == 1: # if one-dimensional
            theta = theta.view(-1, 1)
        if len(x.shape) == 1:
            x = x.view(-1, 1)
        x_dim = self.x_dim
        obs_size = int(x.shape[1] / x_dim)
        score_tensor = self.layers( torch.cat( (theta.repeat_interleave(obs_size, dim = 0), x.reshape(-1, x_dim)), dim = 1 ) ).view(theta.shape[0], obs_size, theta.shape[1])
        return score_tensor



def Like_score_loss(model, theta, x, prop_score, g, g1):
    bias = model(theta, x).mean(dim = 0)
    score = model(theta, x) - bias
    loss1 = torch.norm(score * g(theta, x)**(1/2), dim = -1) ** 2 / 2.
    loss3 = ((score * g(theta, x)) * prop_score).sum(dim = -1)
    
    theta.requires_grad_(True)
    score_tmp = model(theta, x) # In order to calculate grad2
    loss2 = torch.zeros(theta.shape[0]).to(device)
    for i in range(theta.shape[1]):
        grad2 = torch.autograd.grad(outputs = score_tmp[:, i].sum(), inputs = theta, create_graph=True)[0][:, i]
        loss2 += grad2 * (g(theta, x)[:, i]) + score[:, i] * g1(theta, x)[:, i]
    
    loss = loss1 + loss2 + loss3
    return loss.mean() / obs_size, bias
    # return loss.mean(), bias


def train_deb(model, optimizer, dataloader, val_dataloader, g, g1, num_epochs, scheduler, early_stop_patience):
    model.to(device)
    best_val_loss = float('inf')
    best_model_state = None
    best_optimizer_state = None

    # record training loss and validation loss at each epoch and then plot
    start_time = time.time()
    for epoch in range(num_epochs):
        time1 = time.time()
        model.train() 
        total_loss = 0.0
        valid_batches = 0
        for batch_sample in dataloader:
            optimizer.zero_grad()
            batch_theta, batch_x, batch_prop_score = batch_sample
            batch_theta, batch_x, batch_prop_score = batch_theta.to(device), batch_x.to(device), batch_prop_score.to(device)
            loss, bias = Like_score_loss(model, batch_theta, batch_x, batch_prop_score, g, g1)
            if torch.isnan(loss):
                print(f"[WARNING] NaN detected, skipping this minibatch")
                continue
            valid_batches += 1
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        avg_loss = total_loss / valid_batches

        model.eval()
        total_loss_val = 0.0
        val_valid_batches = 0
        for val_batch_sample in val_dataloader:
            val_batch_theta, val_batch_x, val_batch_prop_score = val_batch_sample
            val_batch_theta, val_batch_x, val_batch_prop_score = val_batch_theta.to(device), val_batch_x.to(device), val_batch_prop_score.to(device)
            val_loss, val_bias = Like_score_loss(model, val_batch_theta, val_batch_x, val_batch_prop_score, g, g1)
            if torch.isnan(val_loss):
                print(f"[WARNING] NaN detected, skipping this minibatch")
                continue
            val_valid_batches += 1
            total_loss_val += val_loss.item()    

        avg_val_loss = total_loss_val / val_valid_batches
        if avg_val_loss < best_val_loss:
            best_epoch = epoch + 1
            best_val_loss = avg_val_loss
            best_model_state = copy.deepcopy(model.state_dict())
            best_optimizer_state = copy.deepcopy(optimizer.state_dict())
        
        if scheduler is not None:
            old_lr = optimizer.param_groups[0]['lr']
            scheduler.step(avg_val_loss)
            new_lr = scheduler.get_last_lr()[0]
            if new_lr != old_lr:
                print(f"Epoch {epoch+1}: reducing learning rate to {new_lr:.2e}")
        
        time2 = time.time()
        if epoch % 1 == 0 or epoch == num_epochs:
            print(f'Epoch {epoch+1}/{num_epochs} | Training Loss: {round(avg_loss, 3)} | Validation Loss: {round(avg_val_loss, 3)} | Time: {round(time2 - time1, 2)} seconds')

        # early stop
        if (epoch+1) - best_epoch >= early_stop_patience:
            print(f"Val_loss didn't improve after {early_stop_patience} epochs, stop training")
            break

    # Load best model state after training
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        optimizer.load_state_dict(best_optimizer_state)
        print(f"Return the best model at epoch {best_epoch}, with Validation Loss: {best_val_loss:.3f}")

    
    # output the final model, we just need to minus the bias
    # we calculate the bias using the whole dataset
    total_bias = 0.0 # is actually a vector of the same dimension as theta
    for batch_sample in dataloader:
        batch_theta, batch_x, batch_prop_score = batch_sample
        batch_theta, batch_x, batch_prop_score = batch_theta.to(device), batch_x.to(device), batch_prop_score.to(device)
        loss, bias = Like_score_loss(model, batch_theta, batch_x, batch_prop_score, g, g1)
        total_bias += bias.detach()

    bias_lastlayer = total_bias / len(dataloader)
    end_time = time.time()
    total_duration = end_time - start_time
    print(f'Total training time: {round(total_duration, 2)} seconds')
    return bias_lastlayer


In [ ]:
# No weight function (weight = 1)
def g(theta, x):
    return 1.0 * torch.ones(theta.shape).to(device)

def g1(theta, x):
    return 0.0 * torch.ones(theta.shape).to(device)

## Train the NN

In [ ]:
prop_score_tr = (alpha[:(J-1)] - 1) / theta_tr - ( (alpha[J-1] - 1) / (1- theta_tr.sum(dim = 1)) ).reshape(-1, 1)
prop_score_val = (alpha[:(J-1)] - 1) / theta_val - ( (alpha[J-1] - 1) / (1- theta_val.sum(dim = 1)) ).reshape(-1, 1)

# Create DataLoader
batch_size = 8 
dataset = TensorDataset(theta_tr, x_tr, prop_score_tr)
dataloader = DataLoader(dataset, batch_size = batch_size, shuffle = True)
val_dataset = TensorDataset(theta_val, x_val, prop_score_val)
val_dataloader = DataLoader(val_dataset, batch_size = batch_size, shuffle = False)

In [ ]:
# Create model and optimizer
theta_dim = J - 1
x_dim = J
hidden_size = 64 
num_layers = 2 
learning_rate = 3e-4 
num_epochs = 50

model = ELU_LikeScoreMatchingNN(theta_dim, x_dim, obs_size, hidden_size, num_layers)
optimizer = optim.Adam(model.parameters(), lr = learning_rate, weight_decay = 1e-5)

early_stop_patience = num_epochs # 10
sched_patience = 3
scheduler = None

# train the model
bias_lastlayer = train_deb(model, optimizer, dataloader, val_dataloader, g, g1, num_epochs, scheduler, early_stop_patience)

In [ ]:
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'bias_lastlayer': bias_lastlayer,
}, f'checkpoint_init.pth')

In [ ]:
with torch.no_grad(): 
    model.layers[-1].bias -= bias_lastlayer / obs_size

# Curvature penalty

In [ ]:
def Fisher_penalty(model, theta_extra, x_extra):
    # the same as loss4 function
    # g: weight function, takes input (theta, x), output dimension is the same as theta
    # g1: first derivative of g (the diagonal part, \partial g(\theta, x)_j / \partial \theta_j), output dimension is the same as theta
    # we require g and g1 to be able to address matrix input and do element-wise mapping

    def score_sum_fn(theta_): # in order to calculate Jacobian
        return ( model.cal_penalty(theta_, x_extra).sum(dim = 1) ).sum(dim = 0)
    
    theta_extra.requires_grad_(True)
    obs_size = int(x_extra.shape[1] / model.x_dim) # extra_obs_size
    E_Jacobian = jacobian(score_sum_fn, theta_extra, create_graph = True).permute(1, 0, 2) / obs_size # estimated mean of the Jacobian, of dimension (batch_size, theta_dim, theta_dim)

    score_tensor_extra = model.cal_penalty(theta_extra, x_extra)
    bias_extra_single = ( score_tensor_extra.sum(dim = 1).mean(dim = 0) / obs_size ).unsqueeze(0).unsqueeze(0)
    EssT = torch.einsum('ibc,ibd->icd', score_tensor_extra - bias_extra_single, score_tensor_extra - bias_extra_single) / obs_size 
    
    penalty_fisher = ( (EssT + E_Jacobian)**2 ).sum(dim = (1, 2)) # a vector storing the Fnorm^2 for every row in the minibatch
    
    return penalty_fisher.mean()


def cal_ssT(model, theta_extra, x_extra):
    score_tensor_extra = model.cal_penalty(theta_extra, x_extra)
    obs_size = int(x_extra.shape[1] / model.x_dim) # extra_obs_size
    bias_extra_single = ( score_tensor_extra.sum(dim = 1).mean(dim = 0) / obs_size ).unsqueeze(0).unsqueeze(0)
    EssT = torch.einsum('ibc,ibd->icd', score_tensor_extra - bias_extra_single, score_tensor_extra - bias_extra_single) / obs_size 
    
    return ( EssT**2 ).sum(dim = (1, 2)).mean()

In [ ]:
def train_fisher(model, optimizer, dataloader, val_dataloader, dataloader_extra, val_dataloader_extra, lam_fisher, g, g1, num_epochs, scheduler):
    print(f"Train with penalty lam_fisher = {lam_fisher}")
    model.to(device)
    best_val_sm_loss = float('inf')
    best_model_state = None
    best_optimizer_state = None

    # record training loss and validation loss at each epoch and then plot
    start_time = time.time()
    for epoch in range(num_epochs):
        time1 = time.time()
        model.train() 
        total_loss = 0.0
        total_sm_loss = 0.0
        total_penalty_fisher = 0.0
        
        data_extra_iter = cycle(dataloader_extra)
        valid_batches = 0
        for iter_counter, batch_sample in enumerate(dataloader):
            batch_sample_extra = next(data_extra_iter)
            # print(f"Extra_data_head: {batch_sample_extra[:2]}") # verify the usage is correct
            optimizer.zero_grad()
            batch_theta, batch_x, batch_prop_score = batch_sample
            batch_theta, batch_x, batch_prop_score = batch_theta.to(device), batch_x.to(device), batch_prop_score.to(device)
            
            batch_theta_extra, batch_x_extra = batch_sample_extra
            batch_theta_extra, batch_x_extra = batch_theta_extra.to(device), batch_x_extra.to(device)

            sm_loss, bias = Like_score_loss(model, batch_theta, batch_x, batch_prop_score, g, g1)
            penalty_fisher = Fisher_penalty(model, batch_theta_extra, batch_x_extra)                       
            loss = sm_loss + lam_fisher * penalty_fisher
                
            if torch.isnan(loss):
                print(f"[WARNING] NaN detected, skipping this minibatch")
                continue
            valid_batches += 1
            
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            total_sm_loss += sm_loss.item()
            total_penalty_fisher += penalty_fisher.item()

        model.eval()
        val_total_loss = 0.0
        val_total_sm_loss = 0.0
        val_total_penalty_fisher = 0.0

        val_data_extra_iter = cycle(val_dataloader_extra)
        val_valid_batches = 0
        for val_batch_sample in val_dataloader:
            val_batch_theta, val_batch_x, val_batch_prop_score = val_batch_sample
            val_batch_theta, val_batch_x, val_batch_prop_score = val_batch_theta.to(device), val_batch_x.to(device), val_batch_prop_score.to(device)

            val_batch_sample_extra = next(val_data_extra_iter)
            val_batch_theta_extra, val_batch_x_extra = val_batch_sample_extra
            val_batch_theta_extra, val_batch_x_extra = val_batch_theta_extra.to(device), val_batch_x_extra.to(device)

            val_sm_loss, val_bias = Like_score_loss(model, val_batch_theta, val_batch_x, val_batch_prop_score, g, g1)
            val_penalty_fisher = Fisher_penalty(model, val_batch_theta_extra, val_batch_x_extra)                       
            val_loss = val_sm_loss + lam_fisher * val_penalty_fisher
            
            if torch.isnan(val_loss):
                print(f"[WARNING] NaN detected, skipping this minibatch")
                continue
            val_valid_batches += 1
            
            val_total_loss += val_loss.item()    
            val_total_sm_loss += val_sm_loss.item()
            val_total_penalty_fisher += val_penalty_fisher.item()

        avg_val_sm_loss = val_total_sm_loss / val_valid_batches
        if avg_val_sm_loss < best_val_sm_loss:
            best_epoch = epoch + 1
            best_val_sm_loss = avg_val_sm_loss
            best_model_state = copy.deepcopy(model.state_dict())
            best_optimizer_state = copy.deepcopy(optimizer.state_dict())

        if scheduler is not None:
            old_lr = optimizer.param_groups[0]['lr']
            scheduler.step(val_total_loss / val_valid_batches) # use the penalized loss here
            new_lr = scheduler.get_last_lr()[0]
            if new_lr != old_lr:
                print(f"Epoch {epoch+1}: reducing learning rate to {new_lr:.2e}")
        
        time2 = time.time()
        if epoch % 1 == 0 or epoch == num_epochs:
            print(f'Epoch {epoch+1}/{num_epochs} | Training Loss (Total, SM, pen_fisher): ({total_loss / valid_batches:.3f}, {total_sm_loss / valid_batches:.3f}, {total_penalty_fisher / valid_batches:.3f}) | Validation Loss (Total, SM, pen_fisher): ({val_total_loss / val_valid_batches:.3f}, {val_total_sm_loss / val_valid_batches:.3f}, {val_total_penalty_fisher / val_valid_batches:.3f}). Time: {(time2 - time1):.2f} seconds')


    # Load best model state after training
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        optimizer.load_state_dict(best_optimizer_state)
        print(f"Return the best model at epoch {best_epoch}, with Validation sm Loss: {best_val_sm_loss:.3f}")

    
    # output the final model, we just need to minus the bias
    # we calculate the bias using the whole dataset
    total_bias = 0.0 # is actually a vector of the same dimension as theta
    for batch_sample in dataloader:
        batch_theta, batch_x, batch_prop_score = batch_sample
        batch_theta, batch_x, batch_prop_score = batch_theta.to(device), batch_x.to(device), batch_prop_score.to(device)
        loss, bias = Like_score_loss(model, batch_theta, batch_x, batch_prop_score, g, g1)
        total_bias += bias.detach()


    bias_lastlayer = total_bias / len(dataloader)
    end_time = time.time()
    total_duration = end_time - start_time
    print(f'Total training time: {total_duration/60:.2f} minutes')
    return bias_lastlayer


def check_loss(model, val_dataloader, val_dataloader_extra, lam_fisher):
    model.eval()
    val_total_loss = 0.0
    val_total_sm_loss = 0.0
    val_total_penalty_fisher = 0.0
    val_total_scale = 0.0
    
    val_data_extra_iter = cycle(val_dataloader_extra)
    val_valid_batches = 0
    for val_batch_sample in val_dataloader:
        val_batch_theta, val_batch_x, val_batch_prop_score = val_batch_sample
        val_batch_theta, val_batch_x, val_batch_prop_score = val_batch_theta.to(device), val_batch_x.to(device), val_batch_prop_score.to(device)
        val_batch_sample_extra = next(val_data_extra_iter)
        val_batch_theta_extra, val_batch_x_extra = val_batch_sample_extra
        val_batch_theta_extra, val_batch_x_extra = val_batch_theta_extra.to(device), val_batch_x_extra.to(device)
        val_sm_loss, val_bias = Like_score_loss(model, val_batch_theta, val_batch_x, val_batch_prop_score, g, g1)
        val_penalty_fisher = Fisher_penalty(model, val_batch_theta_extra, val_batch_x_extra)                       
        val_loss = val_sm_loss + lam_fisher * val_penalty_fisher
        val_scale_ssT = cal_ssT(model, val_batch_theta_extra, val_batch_x_extra)
        
        if torch.isnan(val_loss):
            print(f"[WARNING] NaN detected, skipping this minibatch")
            continue
        val_valid_batches += 1
        
        val_total_loss += val_loss.item()    
        val_total_sm_loss += val_sm_loss.item()
        val_total_penalty_fisher += val_penalty_fisher.item()
        val_total_scale += val_scale_ssT.item()
    
    print(f'Validation Loss (Total, SM, pen_fisher): ({val_total_loss / val_valid_batches:.3f}, {val_total_sm_loss / val_valid_batches:.3f}, {val_total_penalty_fisher / val_valid_batches:.3f})')

    print(f'scale E[||EssT||_F^2] = {val_total_scale / val_valid_batches:.3f}')


In [ ]:
batch_size_extra = batch_size
extra_dataset = TensorDataset(theta_tr, x_tr)
extra_dataloader = DataLoader(extra_dataset, batch_size = batch_size_extra, shuffle = True)
extra_val_dataset = TensorDataset(theta_val, x_val)
extra_val_dataloader = DataLoader(extra_val_dataset, batch_size = batch_size_extra, shuffle = False)

In [ ]:
lam_fisher = 0.05 

In [ ]:
# Create model and optimizer
pen_model = ELU_LikeScoreMatchingNN(theta_dim, x_dim, obs_size, hidden_size, num_layers)

# start from the trained model without curvature penalty
checkpoint = torch.load('checkpoint_init.pth', map_location=device)
pen_model.load_state_dict(checkpoint['model_state_dict'])

pen_model.to(device)

# CHECK LOSS
print("Loss of the initial model")
val_sm_loss_init = check_loss(pen_model, val_dataloader, extra_val_dataloader, lam_fisher)
# print("\n")

In [ ]:
lr = 3e-4 
optimizer = optim.Adam(pen_model.parameters(), lr = lr, weight_decay = 1e-5)
num_epochs = 50 

sched_patience = 3
scheduler = None

bias_lastlayer = train_fisher(pen_model, optimizer, dataloader, val_dataloader, extra_dataloader, extra_val_dataloader, lam_fisher, g, g1, num_epochs = num_epochs, scheduler = scheduler)

In [ ]:
torch.save({
    'model_state_dict': pen_model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'bias_lastlayer': bias_lastlayer,
}, f'checkpoint_fisher.pth')

In [ ]:
with torch.no_grad(): 
    pen_model.layers[-1].bias -= bias_lastlayer / obs_size

In [ ]:
# CHECK LOSS
print("Loss of the pen model")
val_sm_loss_init = check_loss(pen_model, val_dataloader, extra_val_dataloader, lam_fisher)
# print("\n")

# Visualizations

## Compare the posterior samples

In [ ]:
# fix theta_star and draw X_n^*
theta_star = 1/J * torch.ones(J).to(device)
MC_size = 10 # number of Xn

# x_obs
x = torch.distributions.Categorical(probs=theta_star).sample((obs_size, MC_size)).T
x_oh = F.one_hot(x, num_classes=J).to(theta_star.dtype)
x = F.one_hot(x, num_classes=J).to(theta_star.dtype).reshape(MC_size, -1)

theta_star = 1/J * torch.ones(J - 1).to(device)

In [ ]:
x_copy = x.clone()

In [ ]:
# draw a set of posteriors based on one observed x
x_sum = x.reshape(MC_size, obs_size, -1).sum(dim = 1) # in order to calculate the true score
num_post = 10000
MC_id = 2
theta_post = torch.distributions.Dirichlet(x_sum[MC_id].cpu() + alpha).sample((num_post,))[:, :-1]
theta_MLE = x_sum[MC_id, :-1].cpu() / obs_size

In [ ]:
r_post = torch.linalg.norm(theta_post - theta_star.cpu(), dim = 1)

In [ ]:
def draw_post(model, x_obs, theta_init, epis = 0.001, S = 100):
    # vectorized, generate multiple MC chains, but only return the last draws of each chain
    # epis: step size
    # S: length of each chain
    # theta_init: dim m*d
    model.to(device)
    model.eval()
    
    theta0 = theta_init.to(device) # initial value of log_theta
    x_obs = x_obs.to(device).view(1, -1)
    for i in tqdm(range(S)):
        like_score_hat = model(theta0, x_obs.repeat(theta0.shape[0], 1)).detach().to(device)
        prior_score = (alpha[:(J-1)].to(device) - 1) / theta0 - ( (alpha[J-1].to(device) - 1) / (1- theta0.sum(dim = 1)) ).reshape(-1, 1)
        theta1 = theta0 + epis * (like_score_hat + prior_score) + np.sqrt(2.0 * epis) * torch.randn(theta0.shape).to(device) # draw a new sample
        theta0 = theta1 # use theta1 as the initial value of the next iteration
    return theta1

In [ ]:
theta_init = 0.05 +  0.45 * torch.rand(5000, 2) 
epis = 0.001 / obs_size 
S = 5000 

est_post = draw_post(model, x[MC_id], theta_init, epis, S)

In [ ]:
pen_est_post = draw_post(pen_model, x[MC_id], theta_init, epis, S)

In [ ]:
# save and plot in R
np.savetxt("x_sum.csv", x_sum[MC_id].detach().cpu().numpy(), delimiter=",")
np.savetxt("est_post.csv", est_post.detach().cpu().numpy(), delimiter=",")
np.savetxt("pen_est_post.csv", pen_est_post.detach().cpu().numpy(), delimiter=",")
np.savetxt("theta_post.csv", theta_post.detach().cpu().numpy(), delimiter=",")

## Error plot

### calculation of the errors

In [ ]:
model.to(device) 
pen_model.to(device)

In [ ]:
def cal_error(model, theta, x):
    # ===== Input ===== #
    # model: the score matching model (n-model)
    # theta: [N, theta_dim]
    # x: [N, x_dim * n], where n is obs_size
    # ===== Output ===== #
    # error: [N, ], each element is ||\hat{s}(theta[i], x[i]) - s^*(theta[i], x[i])||^2
    # error_relative: [N, ], each element is ||\hat{s}(theta[i], x[i]) - s^*(theta[i], x[i])||^2 / ||s^*(theta[i], x[i])||^2
    # scale: [N, ], each element is ||s^*(theta[i], x[i])||^2
    # cos: [N, ], each element is <\hat{s}(theta[i], x[i]), s^*(theta[i], x[i])> / (||\hat{s}(theta[i], x[i])|| \cdot ||s^*(theta[i], x[i])||)
    
    x_sum = x.reshape(x.shape[0], obs_size, -1).sum(dim = 1) # for calculating the true score
    
    score_est = model.cal_penalty(theta, x).detach().sum(dim = 1) # [N, theta_dim]
    score_true = x_sum[:, :-1] / theta - ( x_sum[:, -1] / (1 - theta.sum(dim = 1)) ).reshape(-1, 1) # [N, theta_dim]
    error = torch.linalg.norm(score_est - score_true, dim = 1)**2 # [N, 1]
    scale = torch.linalg.norm(score_true, dim = 1)**2 # [N, 1]
    cos = F.cosine_similarity(score_est, score_true, dim=1)
    return error, scale, cos

In [ ]:
def draw_unif_sphere(theta_star, r, sample_size):
    ### Input:
    # theta_star: [theta_dim, ]
    # r: scalar

    ### Output:
    # theta: [sample_size, theta_dim], each row is iid drawn from a uniform distribution on the sphere of Ball(theta_star, r)

    Z = torch.randn(sample_size, theta_dim).to(theta_star.device)
    Z = Z / torch.linalg.norm(Z, dim = 1, keepdims = True)
    theta = theta_star + r * Z
    return theta

In [ ]:
theta_post = torch.distributions.Dirichlet(x_sum[MC_id].cpu() + alpha).sample((num_post,))[:, :-1]
theta_MLE = x_sum[MC_id, :-1].cpu() / obs_size

In [ ]:
r_set = np.arange(0.0, 0.18, 0.001) 
error_dict = {} # key: r, val: many samples of E_r
error_relative_dict = {}
cos_dict = {}

pen_error_dict = {}
pen_error_relative_dict = {}
pen_cos_dict = {}
for r in r_set:
    error_dict[r] = []
    error_relative_dict[r] = []
    cos_dict[r] = []
    pen_error_dict[r] = []
    pen_error_relative_dict[r] = []
    pen_cos_dict[r] = []

# num_Er: how many theta to draw from Uniform(theta^*, r)
num_Er = 100 

# use the same directions, the plot may look smoother
# draw some random directions of norm 1
Z = torch.randn(num_Er, theta_dim).to(theta_star.device)
Z = Z / torch.linalg.norm(Z, dim = 1, keepdims = True)

for r in tqdm(r_set):
    for j in range(num_Er):
        # draw a theta from the sphere
        theta = theta_star.reshape(1, -1) + r * Z[j].reshape(1, -1)
        theta = theta.repeat(x.shape[0], 1)

        # calculate the errors
        error, scale, cos = cal_error(model, theta, x) 
        pen_error, scale, pen_cos = cal_error(pen_model, theta, x) 

        error_dict[r].append(error.mean().item() / obs_size) # 1/n E||\hat{s} - s||^2
        error_relative_dict[r].append(error.mean().item() / scale.mean().item()) # E[||\hat{s} - s||^2 / ||s||^2]
        cos_dict[r].append(cos.mean().item())
        
        pen_error_dict[r].append(pen_error.mean().item() / obs_size)
        pen_error_relative_dict[r].append(pen_error.mean().item() / scale.mean().item())
        pen_cos_dict[r].append(pen_cos.mean().item())

### calculation of the curvature error

In [ ]:
def curv_error_pertheta(model, theta_extra, x_extra):
    # Input:
    # theta_extra: [N, theta_dim]
    # x_extra: [N, x_dim * n], where n = obs_size, and each row of it is generated by the corresponding row of theta_extra
    # 
    # Output:
    # penalty_fisher: [N, ], each element is the curvature error of the corresponding row of theta_extra
    

    def score_sum_fn(theta_): # in order to calculate Jacobian
        return ( model.cal_penalty(theta_, x_extra).sum(dim = 1) ).sum(dim = 0)

    def true_score_sum_fn(theta_):
        x_sum = x_extra.reshape(-1, obs_size, x_dim).sum(dim = 1)
        return (   x_sum[:, :-1] / theta_ - ( x_sum[:, -1] / (1 - theta_.sum(dim = 1)) ).reshape(-1, 1)   ).sum(dim = 0)
    
    theta_extra.requires_grad_(True)
    obs_size = int(x_extra.shape[1] / model.x_dim) # extra_obs_size
    E_Jacobian = jacobian(score_sum_fn, theta_extra, create_graph = True).permute(1, 0, 2) / obs_size # estimated mean of the Jacobian, of dimension (batch_size, theta_dim, theta_dim)

    E_Jacobian_true = jacobian(true_score_sum_fn, theta_extra, create_graph = True).permute(1, 0, 2) / obs_size

    penalty_fisher = ( (E_Jacobian_true - E_Jacobian)**2 ).sum(dim = (1, 2)) # a vector storing the Fnorm^2 for every row in the minibatch
    scale_fisher = ( E_Jacobian_true**2 ).sum(dim = (1, 2))

    return penalty_fisher.cpu().detach(), scale_fisher.cpu().detach() 

In [ ]:
r_set = np.arange(0.0, 0.2357, 0.002) 
curv_error_dict = {} # key: r, val: many samples of E_r
relative_curv_error_dict = {}

pen_curv_error_dict = {}
pen_relative_curv_error_dict = {}

# num_Er: how many theta to draw from Uniform(theta^*, r)
num_Er = 150 
extra_obs_size = 10000 


# draw some random directions of norm 1
Z = torch.randn(num_Er, theta_dim).to(theta_star.device)
Z = Z / torch.linalg.norm(Z, dim = 1, keepdims = True)

for r in tqdm(r_set):
    # draw {num_Er} theta
    theta = theta_star.reshape(1, -1) + r * Z
    
    # use these theta to generate x, each row of theta corresponds to each row of x
    x = torch.distributions.Categorical(probs=torch.cat((theta, 1-theta.sum(dim = 1, keepdims = True)), dim = 1)).sample((extra_obs_size, )).T
    x = torch.nn.functional.one_hot(x, num_classes=J).to(theta.dtype).reshape(theta.shape[0], -1)
    
    # calculate the errors
    penalty_fisher, scale_fisher = curv_error_pertheta(model, theta, x)
    curv_error_dict[r] = penalty_fisher.clone() # [N, ]
    relative_curv_error_dict[r] = penalty_fisher.clone() / scale_fisher.clone()

    penalty_fisher, scale_fisher = curv_error_pertheta(pen_model, theta, x) 
    pen_curv_error_dict[r] = penalty_fisher.clone() # [N, ]
    pen_relative_curv_error_dict[r] = penalty_fisher.clone() / scale_fisher.clone()

In [ ]:
##### Increase num_Er, in a minibatch way
# have num_Er = 50 * 150 in total
for _ in range(20):
    # num_Er: how many theta to draw from Uniform(theta^*, r)
    num_Er = 150 
    extra_obs_size = 10000 
    

    # draw some random directions of norm 1
    Z = torch.randn(num_Er, theta_dim).to(theta_star.device)
    Z = Z / torch.linalg.norm(Z, dim = 1, keepdims = True)
    
    for r in tqdm(r_set):
        # draw {num_Er} theta
        theta = theta_star.reshape(1, -1) + r * Z
        
        # use these theta to generate x, each row of theta corresponds to each row of x
        x = torch.distributions.Categorical(probs=torch.cat((theta, 1-theta.sum(dim = 1, keepdims = True)), dim = 1)).sample((extra_obs_size, )).T
        x = torch.nn.functional.one_hot(x, num_classes=J).to(theta.dtype).reshape(theta.shape[0], -1)
        
        # calculate the errors
        penalty_fisher, scale_fisher = curv_error_pertheta(model, theta, x)
        curv_error_dict[r] = torch.cat( (curv_error_dict[r], penalty_fisher.clone())  , dim = 0)
        relative_curv_error_dict[r] = torch.cat( (relative_curv_error_dict[r], penalty_fisher.clone() / scale_fisher.clone()) , dim = 0)
    
        penalty_fisher, scale_fisher = curv_error_pertheta(pen_model, theta, x) 
        pen_curv_error_dict[r] = torch.cat( (pen_curv_error_dict[r], penalty_fisher.clone())  , dim = 0)
        pen_relative_curv_error_dict[r] = torch.cat( (pen_relative_curv_error_dict[r], penalty_fisher.clone() / scale_fisher.clone()), dim = 0)

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

mpl.rcParams.update({
    "mathtext.fontset": "stix",
    "font.family": "STIXGeneral",
})


def plot_fan_chart(r_cap, r_set, error_dict, pen_error_dict, qs=(0.025, 0.5, 0.975), ax=None, alpha=0.25, plot_mean=False):
    Q = np.array([np.quantile(np.asarray(error_dict[rr], float), qs) for rr in r_set if rr <= r_cap])
    pen_Q = np.array([np.quantile(np.asarray(pen_error_dict[rr], float), qs) for rr in r_set if rr <= r_cap])

    if ax is None:
        _, ax = plt.subplots()

    x = r_set[r_set <= r_cap]

    if plot_mean:
        mean_err = np.array([np.mean(np.asarray(error_dict[rr], float)) for rr in r_set if rr <= r_cap])
        pen_mean_err = np.array([np.mean(np.asarray(pen_error_dict[rr], float)) for rr in r_set if rr <= r_cap])
        l_plain, = ax.plot(x, mean_err, linewidth=2, label="Plain model")
        l_pen, = ax.plot(x, pen_mean_err, linewidth=2, label="Penalized model")
        b_plain = None
        b_pen = None
    else:
        b_plain = ax.fill_between(x, Q[:, 0], Q[:, 2], alpha=alpha)
        l_plain, = ax.plot(x, Q[:, 1], linewidth=2, label="Plain model")

        b_pen = ax.fill_between(x, pen_Q[:, 0], pen_Q[:, 2], alpha=alpha)
        l_pen, = ax.plot(x, pen_Q[:, 1], linewidth=2, label="Penalized model")

    handles = {
        "plain_line": l_plain,
        "pen_line": l_pen,
        "plain_band": b_plain,
        "pen_band": b_pen,
    }
    return ax, handles


def plot_fan_chart_curv(r_cap, r_set, curv_error_dict, pen_curv_error_dict, qs=torch.tensor([0.025, 0.5, 0.975]), ax=None, alpha=0.25, plot_mean=False):
    Q = torch.stack([torch.quantile(curv_error_dict[rr], qs) for rr in r_set if rr <= r_cap]).cpu().numpy()
    pen_Q = torch.stack([torch.quantile(pen_curv_error_dict[rr], qs) for rr in r_set if rr <= r_cap]).cpu().numpy()

    if ax is None:
        _, ax = plt.subplots()

    x = r_set[r_set <= r_cap]

    if plot_mean:
        mean_err = np.array([curv_error_dict[rr].mean().item() for rr in r_set if rr <= r_cap])
        pen_mean_err = np.array([pen_curv_error_dict[rr].mean().item() for rr in r_set if rr <= r_cap])
        l_plain, = ax.plot(x, mean_err, linewidth=2, label="Plain model")
        l_pen, = ax.plot(x, pen_mean_err, linewidth=2, label="Penalized model")
        b_plain = None
        b_pen = None
    else:
        b_plain = ax.fill_between(x, Q[:, 0], Q[:, 2], alpha=alpha)
        l_plain, = ax.plot(x, Q[:, 1], linewidth=2, label="Plain model")

        b_pen = ax.fill_between(x, pen_Q[:, 0], pen_Q[:, 2], alpha=alpha)
        l_pen, = ax.plot(x, pen_Q[:, 1], linewidth=2, label="Penalized model")

    handles = {
        "plain_line": l_plain,
        "pen_line": l_pen,
        "plain_band": b_plain,
        "pen_band": b_pen,
    }
    return ax, handles

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.patches import Patch

mpl.rcParams.update({
    "mathtext.fontset": "stix",
    "font.family": "STIXGeneral",
})

# =============================
# build ONE figure with two panels
# =============================
r_cap = 0.08
fig, axes = plt.subplots(1, 2, figsize=(10.6, 4.4))   
ax1, ax2 = axes

# ---------- left panel: score error ----------
r_set1 = np.arange(0.0, 0.18, 0.001)
ax1, h1 = plot_fan_chart(
    r_cap, r_set1,
    error_relative_dict, pen_error_relative_dict,
    qs=(1.0, 1.0, 1.0),
    ax=ax1,
    alpha=0.1
)

ax1.set_xlabel(r"$\|\; \theta - \theta^\ast\|$", fontsize=20)
ax1.set_ylabel("Relative score error", fontsize=20)
ax1.tick_params(axis="x", labelsize=14)
ax1.tick_params(axis="y", labelsize=14)
ax1.set_ylim(0, 1)

xmin1 = float(min(np.min(r_set1), np.min(r_post.numpy())))
xmax1 = float(min(max(np.max(r_set1), np.max(r_post.numpy())), r_cap))
ax1.set_xlim(xmin1, xmax1)

ax1_r = ax1.twinx()
ax1_r.set_zorder(0)
ax1.set_zorder(1)
ax1.patch.set_visible(False)

ax1_r.hist(
    r_post,
    bins=30,
    density=True,
    alpha=0.25,
    range=(xmin1, xmax1),
    color="0.75",
    edgecolor="0.35"
)

ax1_r.set_ylabel("", fontsize=16, labelpad=12)
ax1_r.tick_params(axis="y", labelsize=13)

# ---------- right panel: curvature error ----------
r_set2 = np.arange(0.0, 0.2357, 0.002)
ax2, h2 = plot_fan_chart_curv(
    r_cap, r_set2,
    relative_curv_error_dict, pen_relative_curv_error_dict,
    qs=0.98 * torch.tensor([1.0, 1.0, 1.0]),
    ax=ax2,
    alpha=0.1
)

ax2.set_xlabel(r"$\|\; \theta - \theta^\ast\|$", fontsize=20)
ax2.set_ylabel("Relative curvature error", fontsize=20)
ax2.tick_params(axis="x", labelsize=14)
ax2.tick_params(axis="y", labelsize=14)
ax2.set_ylim(0, 1)

xmin2 = float(min(np.min(r_set2), np.min(r_post.numpy())))
xmax2 = float(min(max(np.max(r_set2), np.max(r_post.numpy())), r_cap))
ax2.set_xlim(xmin2, xmax2)

ax2_r = ax2.twinx()
ax2_r.set_zorder(0)
ax2.set_zorder(1)
ax2.patch.set_visible(False)

ax2_r.hist(
    r_post,
    bins=30,
    density=True,
    alpha=0.25,
    range=(xmin2, xmax2),
    color="0.75",
    edgecolor="0.35"
)
ax2_r.set_ylabel(r"Posterior density of $\|\theta-\theta^\ast\|$", fontsize=20, labelpad=12)
ax2_r.tick_params(axis="y", labelsize=13)

# ---------- shared legend at bottom ----------
hist_proxy = Patch(
    facecolor="0.75",
    edgecolor="0.35",
    alpha=0.25,
    label=r"Posterior samples of $\|\theta-\theta^\ast\|$"
)

fig.legend(
    handles=[h1["plain_line"], h1["pen_line"], hist_proxy],
    labels=["Without curvature matching", "With curvature matching", r"Posterior samples of $\|\theta-\theta^\ast\|$"],
    loc="lower center",
    bbox_to_anchor=(0.5, -0.08),   
    ncol=3,
    frameon=False,
    fontsize=16,
    handlelength=2.2,
    columnspacing=1.5
)

# ---------- layout ----------
fig.subplots_adjust(
    left=0.08,
    right=0.92,
    top=0.98,
    bottom=0.2,
    wspace=0.4
)

# fig.savefig("curv_plots/error_score_and_curv.pdf", bbox_inches="tight", format="pdf", dpi=300)
plt.show()

### 2-d heatmap of one dataset

In [ ]:
# draw a set of posteriors based on one observed x
MC_size = 10
x_sum = x_copy.reshape(MC_size, obs_size, -1).sum(dim = 1) # in order to calculate the true score
MC_id = 2
theta_MLE = x_sum[MC_id, :-1].cpu() / obs_size

In [ ]:
theta_MLE

In [ ]:
# ranges + resolution
x1_min, x1_max = 0.25, 0.4
x2_min, x2_max = 0.25, 0.4
n1, n2 = 200, 200

x1 = torch.linspace(x1_min, x1_max, n1, device=device)
x2 = torch.linspace(x2_min, x2_max, n2, device=device)

# meshgrid -> (n2, n1)
X1, X2 = torch.meshgrid(x1, x2, indexing="xy")

# stack into points -> (n2, n1, 2) -> flatten to (N, 2)
theta = torch.stack([X1, X2], dim=-1).reshape(-1, 2)

In [ ]:
error = torch.zeros(theta.shape[0])
scale = torch.zeros(theta.shape[0])
bsize = 1000
for start in tqdm(range(0, theta.shape[0], bsize)):
    theta_tmp = theta[start:(start+bsize)]
    error_tmp, scale_tmp, _ = cal_error(model, theta_tmp, x_copy[MC_id].unsqueeze(0).expand(theta_tmp.shape[0], -1))
    error[start:(start+bsize)] = error_tmp
    scale[start:(start+bsize)] = scale_tmp
error /= obs_size
scale /= obs_size

In [ ]:
pen_error = torch.zeros(theta.shape[0])
bsize = 1000
for start in tqdm(range(0, theta.shape[0], bsize)):
    theta_tmp = theta[start:(start+bsize)]
    error_tmp, _, _ = cal_error(pen_model, theta_tmp, x_copy[MC_id].unsqueeze(0).expand(theta_tmp.shape[0], -1))
    pen_error[start:(start+bsize)] = error_tmp
pen_error /= obs_size

In [ ]:
error_post, scale_post, _ = cal_error(model, theta_post.to(device), x_copy[MC_id].unsqueeze(0).expand(theta_post.shape[0], -1))
error_post /= obs_size
scale_post /= obs_size

In [ ]:
rela_error = error / (scale + scale_post.mean().cpu())
pen_rela_error = pen_error / (scale + scale_post.mean().cpu())

In [ ]:
import pandas as pd

theta_np = theta.detach().cpu().numpy()
err_np = error.detach().cpu().numpy().reshape(-1)
pen_err_np = pen_error.detach().cpu().numpy().reshape(-1)

df = pd.DataFrame({
    "theta1": theta_np[:, 0],
    "theta2": theta_np[:, 1],
    "rela_error": rela_error,
    "pen_rela_error": pen_rela_error
})

df.to_csv("rela_error_1data.csv", index=False)